In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType
import os
from pyspark.sql.functions import explode
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder.appName("imdb").getOrCreate()

In [4]:
spark

In [5]:
df = spark.read.json("raw_movies (1).jsonl")
df.printSchema()

root
 |-- metadata: struct (nullable = true)
 |    |-- entity: string (nullable = true)
 |    |-- extraction_method: string (nullable = true)
 |    |-- http_status: long (nullable = true)
 |    |-- ingestion_id: string (nullable = true)
 |    |-- ingestion_timestamp: string (nullable = true)
 |    |-- raw_hash: string (nullable = true)
 |    |-- search_parameters: struct (nullable = true)
 |    |    |-- movie_count: long (nullable = true)
 |    |-- source_system: string (nullable = true)
 |-- raw_payload: struct (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- duration_seconds: long (nullable = true)
 |    |-- genres: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- imdb_id: string (nullable = true)
 |    |-- rating: double (nullable = true)
 |    |-- release_date: struct (nullable = true)
 |    |    |-- day: long (nullable = true)
 |    |    |-- month: long (nullable = true)
 |    |    |-- year: long (nullable = true)
 |

In [6]:
transform_df = df.select(
    F.col("metadata.ingestion_id").alias("ingestion_id"),
    F.col("metadata.source_system").alias("source_system"),
    F.col("metadata.ingestion_timestamp").cast("timestamp").alias("ingestion_timestamp"),
    F.col("metadata.raw_hash").alias("raw_hash"),
    F.col("raw_payload.imdb_id").alias("imdb_id"),
    F.col("raw_payload.title").alias("title"),
    F.make_date(
    F.col("raw_payload.release_date.year"),
    F.col("raw_payload.release_date.month"),
    F.col("raw_payload.release_date.day"),
    ).alias("release_date"),
    F.col("raw_payload.genres").alias("genres"),
    F.col("raw_payload.description").alias("description"),
    (F.col("raw_payload.duration_seconds")).alias("duration_seconds"),
    F.col("raw_payload.rating").cast("double").alias("imdb_rating"),
    F.col("raw_payload.vote_count").cast("long").alias("imdb_vote_count"),
    ).filter(F.col("imdb_id").isNotNull())

transform_df.show()

+--------------------+-------------+--------------------+--------------------+----------+--------------------+------------+--------------------+--------------------+----------------+-----------+---------------+
|        ingestion_id|source_system| ingestion_timestamp|            raw_hash|   imdb_id|               title|release_date|              genres|         description|duration_seconds|imdb_rating|imdb_vote_count|
+--------------------+-------------+--------------------+--------------------+----------+--------------------+------------+--------------------+--------------------+----------------+-----------+---------------+
|1242ddb7-280f-429...|         imdb|2026-03-28 15:33:...|de3fd106e064e7e37...|tt12042730|   Project Hail Mary|  2026-03-20|[Drama, Sci-Fi, T...|A science teacher...|            9360|        8.5|          87166|
|25410c54-a599-49e...|         imdb|2026-03-28 15:33:...|d6484a45faf555536...|tt15574124|Peaky Blinders: T...|  2026-03-20|[Crime, Drama, Hi...|During World

In [7]:
file_path = "part-00000-3a3ddf63-2f03-46b4-be79-92b1179b6261-c000.snappy.parquet"
df = spark.read.parquet(file_path)
df.show(5, truncate=False)

+------------------------------------+-------------+--------------------------+----------------------------------------------------------------+---------+-------------------------------------------------+--------------------------------+------------+----------------+-----------+---------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ingestion_id                        |source_system|ingestion_timestamp       |raw_hash                                                        |imdb_id  |title                                            |genres                          |release_date|duration_seconds|imdb_rating|imdb_vote_count|description                                                                                                                                                                                   |
+-------------------------

In [8]:
df2 = spark.read.json("raw_reviews_110227_part_1.jsonl")
clean_df = df2.select("raw_payload.*")
df2.printSchema()

root
 |-- metadata: struct (nullable = true)
 |    |-- entity: string (nullable = true)
 |    |-- extraction_method: string (nullable = true)
 |    |-- http_status: long (nullable = true)
 |    |-- ingestion_id: string (nullable = true)
 |    |-- ingestion_timestamp: string (nullable = true)
 |    |-- raw_hash: string (nullable = true)
 |    |-- search_parameters: struct (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- movie_id: string (nullable = true)
 |    |-- source_system: string (nullable = true)
 |-- raw_payload: struct (nullable = true)
 |    |-- author: string (nullable = true)
 |    |-- content: string (nullable = true)
 |    |-- date: string (nullable = true)
 |    |-- movie_id: string (nullable = true)
 |    |-- rating: long (nullable = true)
 |    |-- title: string (nullable = true)



In [9]:
clean_df.show(5, truncate=False)

+------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
imdb_df = spark.read.parquet("28/part-00000-69827c85-a333-4393-9c2a-78930da9694b-c000.snappy.parquet")
imdb_df.show(100)

+--------------------+-------------+--------------------+--------------------+-------+----------+--------------------+------+--------------------+-------------------+
|        ingestion_id|source_system| ingestion_timestamp|           review_id|tmdb_id|   imdb_id|              author|rating|             content|         created_at|
+--------------------+-------------+--------------------+--------------------+-------+----------+--------------------+------+--------------------+-------------------+
|b47a4a1b-27fb-42c...|         imdb|2026-03-28 15:35:...|000524abf6de59d91...|   None|tt27543632|       Maoz_Shechter|   7.0|For the first hou...|2026-01-13 00:00:00|
|3bc71b6a-20fb-4f0...|         imdb|2026-03-28 15:38:...|002c15fdae45738c4...|   None|tt15239678|         madanmarwah|   9.0|This movie is a s...|2024-04-24 00:00:00|
|cf86d66d-e1a4-48a...|         imdb|2026-03-28 15:38:...|0033bf8a018dd3d8f...|   None|tt14205554|         zorox-64846|  10.0|One of the best m...|2025-07-05 00:00:00